[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [7]:
class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
      #same kv in attention with more queries --> n_q = n_heads, while n_kv = num_kv_heads
      self.d_model = d_model
      self.num_heads = num_heads
      self.num_kv_heads = num_kv_heads

      self.head_dim = self.d_model // num_heads
      self.kv_per_q = num_heads // num_kv_heads

      self.W_q = nn.Linear(d_model, d_model)
      self.W_k = nn.Linear(d_model, d_model // self.kv_per_q)
      self.W_v = nn.Linear(d_model, d_model // self.kv_per_q)
      self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
      """
      x: (B, S, d_model)

      return: (B, S, d_model)
      """
      B, S, _ = x.shape

      Q = self.W_q(x).reshape(B, S, self.num_heads, self.head_dim).permute(0, 2, 1, 3) #(B, num_heads, S, head_dim)
      K = self.W_k(x).reshape(B, S, self.num_kv_heads, self.head_dim).permute(0, 2, 1, 3) #(B, num_kv_heads, S, head_dim)
      V = self.W_v(x).reshape(B, S, self.num_kv_heads, self.head_dim).permute(0, 2, 1, 3) #(B, num_kv_heads, S, head_dim)

      K = torch.repeat_interleave(K, self.kv_per_q, dim = 1) #(B, num_heads, S, head_dim)
      V = torch.repeat_interleave(V, self.kv_per_q, dim = 1) #(B, num_heads, S, head_dim)

      d_k = K.shape[-1]

      att_weights = torch.softmax(Q @ K.permute(0, 1, 3, 2) / (d_k ** 0.5), dim = -1) #(B, num_heads, S, S)

      hidden = att_weights @ V #(B, num_heads, S, head_dim)
      hidden = hidden.permute(0, 2, 1, 3).reshape(B, S, self.d_model)

      return self.W_o(hidden) #(B, S, d_model)


In [8]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [9]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (6.9ms)
  ✅ [2/5] nn.Linear with correct shapes (0.9ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (3.1ms)
  ✅ [4/5] KV heads are shared correctly (11.1ms)
  ✅ [5/5] Gradient flow (53.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (75.4ms total)
  Progress saved. Run status() to see your dashboard.

